# Phase D — Final frozen-candidate evaluation

This notebook reads the saved Phase D outputs. It does not train or tune a model.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
table_dir = project_root / 'reports' / 'tables'
figure_dir = project_root / 'reports' / 'figures' / 'final'

metrics = json.loads((table_dir / 'final_holdout_metrics.json').read_text())
classification = pd.read_csv(table_dir / 'final_holdout_classification.csv')
regression = pd.read_csv(table_dir / 'final_holdout_regression.csv')
economics = pd.read_csv(table_dir / 'final_holdout_economics.csv')
importance = pd.read_csv(table_dir / 'final_holdout_feature_importance.csv')
simulation = pd.read_csv(table_dir / 'final_holdout_lightgbm_simulation.csv')


## Frozen configuration


In [ ]:
pd.Series(metrics['configuration'], name='value').to_frame()


## Classification comparison


In [ ]:
classification[['model', 'accuracy', 'balanced_accuracy', 'macro_f1', 'down_recall', 'flat_recall', 'up_recall']]


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.bar(classification['model'], classification['balanced_accuracy'])
plt.axhline(1/3, linestyle='--', linewidth=1)
plt.ylabel('Balanced accuracy')
plt.title('Final classification comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## Return-prediction comparison


In [ ]:
regression[['model', 'mae_bps', 'mae_improvement_pct_vs_zero', 'rank_ic', 'nonzero_directional_accuracy']]


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.bar(regression['model'], regression['rank_ic'])
plt.axhline(0, linewidth=1)
plt.ylabel('Spearman rank IC')
plt.title('Final return-ranking comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## Frozen-threshold economics


In [ ]:
economics[['model', 'active_signal_fraction', 'mean_gross_return_active_bps', 'mean_estimated_cost_active_bps', 'mean_net_return_active_bps', 'break_even_cost_fraction', 'max_drawdown_bps']]


In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(simulation['time_seconds'], simulation['cumulative_net_bps'])
plt.axhline(0, linewidth=1)
plt.xlabel('Seconds from midnight')
plt.ylabel('Cumulative net return (bps)')
plt.title('Frozen LightGBM cumulative net signal return')
plt.tight_layout()
plt.show()


## Final feature importance


In [ ]:
top = (importance[importance['model'] == 'lightgbm_classifier']
       .nlargest(15, 'normalized_gain_importance'))
top[['feature', 'normalized_gain_importance']]


In [ ]:
plt.figure(figsize=(9, 6))
plt.barh(top['feature'][::-1], top['normalized_gain_importance'][::-1])
plt.xlabel('Normalised gain importance')
plt.title('Frozen LightGBM classifier feature importance')
plt.tight_layout()
plt.show()


## Interpretation checklist

- Did LightGBM remain above the linear comparator on balanced accuracy and rank IC?
- Did it beat the zero-return MAE baseline?
- Is gross edge positive at the frozen threshold?
- Does gross edge cover the full estimated quoted-spread cost?
- Do not tune the model or threshold from this result.
